In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/amankharwal/Website-data/refs/heads/master/weatherAUS.csv')
df.head()

,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RISK_MM,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,0.0,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,0.0,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,0.0,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,1.0,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,0.2,No


In [3]:
df['RainTomorrow'] = df['RainTomorrow'].map({'Yes': 1, 'No': 0})

In [4]:
df = df.dropna(subset=['RainTomorrow'])

In [5]:
X = df.drop('RainTomorrow', axis=1)
y = df['RainTomorrow']

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [7]:
numerical_features = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()

In [8]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, numerical_features),
    ('cat', cat_pipeline, categorical_features)
])

In [9]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

In [10]:
param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [10, 20, None]
}

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring='accuracy',
    verbose=2
)

grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 6 candidates, totalling 30 fits
[CV] END classifier__max_depth=10, classifier__n_estimators=100; total time=   4.3s
[CV] END classifier__max_depth=10, classifier__n_estimators=100; total time=   4.0s
[CV] END classifier__max_depth=10, classifier__n_estimators=100; total time=   3.8s
[CV] END classifier__max_depth=10, classifier__n_estimators=100; total time=   4.6s
[CV] END classifier__max_depth=10, classifier__n_estimators=100; total time=   4.6s
[CV] END classifier__max_depth=10, classifier__n_estimators=200; total time=   8.1s
[CV] END classifier__max_depth=10, classifier__n_estimators=200; total time=   7.9s
[CV] END classifier__max_depth=10, classifier__n_estimators=200; total time=   8.3s
[CV] END classifier__max_depth=10, classifier__n_estimators=200; total time=   8.7s
[CV] END classifier__max_depth=10, classifier__n_estimators=200; total time=   8.7s
[CV] END classifier__max_depth=20, classifier__n_estimators=100; total time=  23.4s
[CV] END classif

In [ ]:
best_model = grid_search.best_estimator_

In [ ]:
test_score = best_model.score(X_test, y_test)
print(f"Accuracy: {test_score:.3f}")

In [ ]:
y_pred = best_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

TP = cm[1,1]
FN = cm[1,0]

tpr = TP / (TP + FN)
print(f"True Positive Rate: {tpr*100:.0f}%")

In [ ]:
pipeline.set_params(classifier=LogisticRegression(max_iter=1000, random_state=42))
grid_search.estimator = pipeline

grid_search.fit(X_train, y_train)

log_model = grid_search.best_estimator_

print("Logistic Accuracy:", log_model.score(X_test, y_test))

In [ ]:
rf = best_model.named_steps['classifier']
importances = rf.feature_importances_

feature_names = best_model.named_steps['preprocessor'].get_feature_names_out()

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

importance_df.head(10)